In [1]:
import json
from pathlib import Path

def generate_html_dashboard_from_analysis_obj(analysis_obj, out_file="dashboard_preview.html"):
    """
    Prend l'objet analysis_obj (dict Python) et génère un HTML lisible
    listant sheets + visuals (comme ton screenshot).
    """
    definition = analysis_obj.get("Definition", {})
    sheets = definition.get("Sheets", [])

    html = []
    html.append("<html><head><meta charset='utf-8'>")
    html.append("<style>")
    html.append("body{font-family:Arial, sans-serif; margin:24px;}")
    html.append("h1{margin-bottom:10px;}")
    html.append("h2{margin-top:30px;}")
    html.append(".card{border:1px solid #e5e5e5; border-radius:8px; padding:14px; margin:12px 0;}")
    html.append(".small{color:#444; font-size:13px;}")
    html.append("</style></head><body>")
    html.append("<h1>ESG Dashboard – Aperçu local</h1>")

    for sheet in sheets:
        sheet_name = sheet.get("Name", "Sheet")
        html.append(f"<h2>{sheet_name}</h2>")

        visuals = sheet.get("Visuals", [])
        for i, v in enumerate(visuals, start=1):
            
            if isinstance(v, dict):
                visual_type = next(iter(v.keys()), "Unknown")
                visual_obj = v.get(visual_type, {})
            else:
                visual_type = "Unknown"
                visual_obj = {}

            title = "HIDDEN"
            # Essaye de trouver un titre
            try:
                title_obj = visual_obj.get("Title", {})
                if isinstance(title_obj, dict):
                    if title_obj.get("Visibility") == "VISIBLE":
                        title_fmt = title_obj.get("FormatText", {})
                        if isinstance(title_fmt, dict):
                            plain = title_fmt.get("PlainText", "")
                            if plain:
                                title = plain
            except Exception:
                pass

            # VisualId 
            visual_id = visual_obj.get("VisualId", f"visual_{i}")

            html.append("<div class='card'>")
            html.append(f"<b>{title}</b><br>")
            html.append(f"<div class='small'>Type : {visual_type}</div>")
            html.append(f"<div class='small'>VisualId : {visual_id}</div>")
            html.append("</div>")

    html.append("</body></html>")

    Path(out_file).write_text("\n".join(html), encoding="utf-8")
    return out_file


def save_analysis_json(analysis_obj, out_file="esg_analysis.json"):
    Path(out_file).write_text(json.dumps(analysis_obj, indent=2), encoding="utf-8")
    return out_file
